In [1]:
!pip install transformers accelerate datasets scikit-learn

In [2]:
!nvidia-smi

Tue Apr  7 16:15:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             26W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

import warnings
warnings.filterwarnings('ignore')

# Kiểm tra xem GPU đã sẵn sàng chưa
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng thiết bị: {device}")

Đang sử dụng thiết bị: cuda


In [4]:
import re

train_path = '/kaggle/input/competitions/midtermNLP01/train.csv'
test_path = '/kaggle/input/competitions/midtermNLP01/test.csv'

# Load dữ liệu
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Hàm làm sạch nhẹ
def clean_text(text):
    if not isinstance(text, str): return ""
    text = str(text).lower()

    # 1. Dịch ngược Acronyms cảm xúc thành tiếng Việt rõ ràng
    emoji_dict = {
        r'\bcolonsmile\b': 'rất vui',
        r'\bcolonsad\b': 'rất buồn',
        r'\bcolonsurprise\b': 'rất ngạc nhiên',
        r'\bcolonlove\b': 'rất yêu',
        r'\bcolonsmilesmile\b': 'cực kỳ vui',
        r'\bcoloncontemn\b': 'đáng khinh',
        r'\bcolonbigsmile\b': 'cười lớn',
        r'\bcoloncc\b': 'bực bội',
        r'\bcolonsmallsmile\b': 'mỉm cười',
        r'\bcoloncolon\b': '', # Bỏ qua
        r'\bcolonlovelove\b': 'cực kỳ yêu',
        r'\bcolonhihi\b': 'vui vẻ',
        r'\bdoubledot\b': '', # Bỏ qua
        r'\bcolonsadcolon\b': 'buồn khóc',
        r'\bcolondoublesurprise\b': 'cực kỳ ngạc nhiên'
    }
    for k, v in emoji_dict.items():
        text = re.sub(k, v, text)

    # 2. Chuẩn hóa Teencode
    teencode_dict = {
        r'\bko\b': 'không', r'\bk\b': 'không', r'\bkhong\b': 'không',
        r'\bdc\b': 'được', r'\bđc\b': 'được',
        r'\bvs\b': 'với', r'\bgv\b': 'giảng viên',
        r'\bsv\b': 'sinh viên', r'\bth\b': 'thực hành',
        r'\bbt\b': 'bài tập', r'\bđh\b': 'đại học',
        r'\bhok\b': 'học', r'\bđk\b': 'được'
    }
    for k, v in teencode_dict.items():
        text = re.sub(k, v, text)

    # Xóa ký tự đặc biệt không cần thiết nhưng giữ lại dấu chấm, phẩy
    text = re.sub(r'[^\w\s\.\,]', '', text)

    return " ".join(text.split())




In [5]:
train_df['clean_sentence'] = train_df['sentence'].apply(clean_text)
test_df['clean_sentence'] = test_df['sentence'].apply(clean_text)

# Chia tập Train thành Train (85%) và Validation (15%) để đánh giá sớm
X_train, X_val, y_train, y_val = train_test_split(
    train_df['clean_sentence'].tolist(),
    train_df['sentiment'].tolist(),
    test_size=0.15,
    random_state=42,
    stratify=train_df['sentiment'] # Đảm bảo tỷ lệ nhãn được cân bằng
)

print(f"Số lượng tập Train: {len(X_train)} | Tập Validation: {len(X_val)}")

Số lượng tập Train: 9623 | Tập Validation: 1699


In [6]:
# Tải Tokenizer của PhoBERT 
model_name = "vinai/phobert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize dữ liệu Train và Validation
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=64)
val_encodings = tokenizer(X_val, truncation=True, padding=True, max_length=64)
test_encodings = tokenizer(test_df['clean_sentence'].tolist(), truncation=True, padding=True, max_length=64)


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
# Tạo class Dataset chuẩn của PyTorch
class StudentSentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings.input_ids)

# Khởi tạo đối tượng Dataset
train_dataset = StudentSentimentDataset(train_encodings, y_train)
val_dataset = StudentSentimentDataset(val_encodings, y_val)
test_dataset = StudentSentimentDataset(test_encodings)

print("Đã Tokenize và chuẩn bị xong Dataset!")

Đã Tokenize và chuẩn bị xong Dataset!


In [8]:
# Hàm tính Macro F1-score để HuggingFace Trainer tự động tính toán
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {'macro_f1': f1, 'accuracy': acc}

# Khởi tạo Mô hình phân loại 3 nhãn (0, 1, 2)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
model.to(device)
print("Load mô hình PhoBERT thành công!")

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load mô hình PhoBERT thành công!


In [9]:
import torch
from torch import nn
from transformers import Trainer, TrainingArguments
from sklearn.utils.class_weight import compute_class_weight

# 1. Tính toán Class Weights dựa trên phân phối dữ liệu thực tế
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
# Chuyển thành Tensor và đẩy lên GPU
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print(f"Trọng số các nhãn (0, 1, 2): {class_weights}")



# 2. Tạo Custom Trainer để sử dụng Class Weights
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Sử dụng CrossEntropyLoss kết hợp trọng số
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

        
# 3. Cấu hình Training (Giảm Learning Rate xuống một chút để mô hình hội tụ sâu hơn)
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=1e-5,
    weight_decay=0.02,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir='./logs',
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=True,
    dataloader_num_workers=2
)



`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Trọng số các nhãn (0, 1, 2): [0.72212217 7.52973396 0.67458815]


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

In [10]:
# 4. Sử dụng WeightedLossTrainer
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Bắt đầu Fine-tuning PhoBERT với Custom Loss...")
trainer.train()

Bắt đầu Fine-tuning PhoBERT với Custom Loss...


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.609965,0.554502,0.829531,0.931725
2,0.433466,0.688286,0.822449,0.936433
3,0.359005,0.853662,0.816690,0.934667
4,0.335666,0.751509,0.824675,0.935256
5,0.258485,0.780200,0.823934,0.934667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=3010, training_loss=0.37341920735432066, metrics={'train_runtime': 355.1353, 'train_samples_per_second': 135.484, 'train_steps_per_second': 8.476, 'total_flos': 1582462761747840.0, 'train_loss': 0.37341920735432066, 'epoch': 5.0})

In [11]:
print("Đang dự đoán trên tập test...")
predictions = trainer.predict(test_dataset)
predicted_labels = np.argmax(predictions.predictions, axis=1)

submission = pd.DataFrame({
    'id': test_df['id'],
    'sentiment': predicted_labels
})

submission_path = '/kaggle/working/submission.csv'

submission.to_csv(submission_path, index=False)

print(f"{submission_path}")

print(submission['sentiment'].value_counts())

Đang dự đoán trên tập test...


/kaggle/working/submission.csv
sentiment
2    2423
0    2231
1     199
Name: count, dtype: int64
